## XRD CNN — Interpretability Explorer

**Workflow:**
1. Run the **Utility Code** section at the bottom once (or after kernel restart).
2. Set `index` in Cell 1.
3. Run Cell 2 to load the sample and see the prediction.
4. Run Cell 3 to see all four visualisations.

In [2]:
# ══════════════════════════════════════════════════════════
#  ► CHANGE THIS INDEX to explore a different sample
# ══════════════════════════════════════════════════════════
index = 26

In [3]:
import pickle

# ── Load structure from DB ────────────────────────────────────────────────
conn = sqlite3.connect('intrep_crystal_xrd.db')
cursor = conn.cursor()
cursor.execute(
    'SELECT * FROM Structures WHERE structure_id = ?',
    (hdf5_to_structure[index],),
)
row = cursor.fetchone()
conn.close()

spacegroup_number = row[5]
unit_cell = (row[7], row[8], row[9], row[10], row[11], row[12])

# ── Compute diffraction pattern ────────────────────────────────────────────
twotheta_pattern, intensity_pattern = generate_diffraction_pattern_helper(
    spacegroup_number=spacegroup_number,
    unit_cell=unit_cell, wavelength=1.541838,
    twotheta_min=5.0, twotheta_max=90.0, twothstep=0.01,
    u=0.01, v=-0.01, w=0.02, peak_scale=1000,
)
two_theta = np.linspace(5.0, 90.0, len(intensity_pattern))
twotheta_peaks, hkls = get_reflections_with_hkl(spacegroup_number, unit_cell)

# ── Prepare model input ────────────────────────────────────────────────────
intensity_padded = np.zeros(spec_length, dtype=np.float32)
copy_len = min(len(intensity_pattern), spec_length)
intensity_padded[:copy_len] = intensity_pattern[:copy_len]
spectrum = (
    torch.tensor(intensity_padded, dtype=torch.float32)
    .unsqueeze(0)   # shape: (1, spec_length)
    .to(device)
)

# ── Forward pass ──────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    logits = model(spectrum)
pred_class = logits.argmax(dim=-1).item()

# ── Resolve true label ────────────────────────────────────────────────────
with open('SG_To_Ext_Map.pkl', 'rb') as f:
    sg_to_ext = pickle.load(f)
true_ext = sg_to_ext.get(spacegroup_number, '?')
true_ext = true_ext-1
# ── Print result ──────────────────────────────────────────────────────────
correct = (pred_class == true_ext)
status  = '✓  CORRECT' if correct else '✗  WRONG'
print(f"{'─' * 50}")
print(f'  Sample index     : {index}')
print(f'  Structure ID     : {row[0]}')
print(f'  Crystal system   : {row[4]}')
print(f'  Space group      : {spacegroup_number}')
print(f'  True ext. group  : {true_ext}  (raw DB field: {row[3]})')
print(f'  Predicted class  : {pred_class}')
print(f'  Result           : {status}')
print(f"{'─' * 50}")


NameError: name 'sqlite3' is not defined

In [4]:
from captum.attr import LayerGradCam

# ── Saliency ─────────────────────────────────────────────────────────────
def compute_saliency(model, x, class_idx=None):
    """
    Vanilla input-gradient saliency.

    Fixes vs original:
      - model.zero_grad() called BEFORE the forward pass, not after.
      - x is detached before re-enabling grad so no upstream graph leaks.
    """
    model.eval()
    x = x.clone().detach().requires_grad_(True)
    model.zero_grad()                          # zero BEFORE forward
    logits = model(x)
    if class_idx is None:
        class_idx = logits.argmax(dim=-1).item()
    logits[0, class_idx].backward()
    return x.grad[0].abs().cpu().numpy(), class_idx


# ── GradCAM ──────────────────────────────────────────────────────────────
def compute_gradcam(model, x, target_layer, class_idx=None):
    """
    GradCAM via Captum.

    IMPORTANT — target_layer should be the LAST spatial layer before the
    classifier (e.g. model.resnet.layer4[-1]), NOT the first conv.
    Using the first conv gives very coarse, semantically weak maps.
    To inspect layer names: print([n for n, _ in model.named_modules()])

    Fixes vs original:
      - Applies ReLU (np.maximum(raw, 0)) instead of np.abs.
        ReLU keeps only activations that positively support the predicted
        class; abs treats inhibitory and excitatory regions the same, which
        distorts attribution semantics.
      - Removed the redundant triple-interpolation loop from the original.
    """
    model.eval()
    with torch.no_grad():
        if class_idx is None:
            class_idx = model(x).argmax(dim=-1).item()
    lgc = LayerGradCam(model, target_layer)
    with torch.enable_grad():
        attrs = lgc.attribute(x, target=class_idx)
    raw = attrs[0, 0].detach().cpu().numpy()
    return np.maximum(raw, 0), class_idx   # ReLU — standard GradCAM


# ── Shared helpers ────────────────────────────────────────────────────────
def prepare_attn(raw, target_len):
    """Upsample to target_len, then percentile-clip + normalize to [0,1]."""
    profile = np.interp(
        np.linspace(0, 1, target_len),
        np.linspace(0, 1, len(raw)),
        raw,
    )
    low  = np.percentile(profile, 80)
    high = profile.max()
    profile = np.clip(profile, low, high)
    return (profile - low) / (high - low + 1e-8)


def draw_xrd_attn(ax, two_theta, intensity, attn_profile, peaks, hkls,
                  n_labels=10, title=''):
    """
    Draw XRD + attention heatmap + HKL annotations into an existing Axes.
    n_labels=0  → label every HKL
    n_labels>0  → label the N highest-intensity peaks only
    """
    peak_max = intensity.max()
    y_max    = peak_max * 1.30

    if attn_profile is not None:
        a_min, a_max = attn_profile.min(), attn_profile.max()
        attn_norm = (attn_profile - a_min) / (a_max - a_min + 1e-8)
        bin_edges = np.linspace(two_theta[0], two_theta[-1], 51)
        bin_idx   = np.digitize(two_theta, bin_edges) - 1
        for i in range(50):
            mask = bin_idx == i
            if np.any(mask):
                ax.fill_between(
                    two_theta[mask], 0, y_max,
                    color=(1.0, 0.0, 0.0, float(attn_norm[mask].mean())),
                    step='mid',
                )

    ax.plot(two_theta, intensity, 'k-', linewidth=0.8)

    peak_int = np.interp(peaks, two_theta, intensity)
    top_idx  = (np.arange(len(hkls)) if n_labels == 0
                else np.argsort(peak_int)[-n_labels:])

    placed = []
    y_step = 0.06 * peak_max
    for i in top_idx:
        h, k, l = hkls[i]
        x      = peaks[i]
        base_y = peak_int[i] + 0.02 * peak_max
        y      = base_y
        for px, py in placed:
            if abs(x - px) < 1.0:
                y = max(y, py + y_step)
        placed.append((x, y))
        ax.annotate(
            f'({h}{k}{l})',
            xy=(x, peak_int[i]), xytext=(x, y),
            xycoords='data', textcoords='data',
            ha='center', va='bottom', fontsize=6, rotation=45,
            arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
        )

    ax.set_ylim(0, y_max)
    ax.set_xlabel('2θ (degrees)', fontsize=8)
    ax.set_ylabel('Intensity', fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=7)


# ── Run saliency ─────────────────────────────────────────────────────────
with torch.enable_grad():
    sal_raw, _ = compute_saliency(model, spectrum)
sal_profile = prepare_attn(sal_raw, len(intensity_pattern))

# ── Run GradCAM ──────────────────────────────────────────────────────────
# ⚠  Verify this is the correct last conv/residual layer for your architecture.
#   Recommended: model.resnet.layer4[-1]  (last residual stage)
#   Avoid:       model.resnet.conv        (first conv — NOT suitable for GradCAM)
target_layer = model.resnet.layer4[-1]
gcam_raw, _  = compute_gradcam(model, spectrum, target_layer)
gcam_profile = prepare_attn(gcam_raw, len(intensity_pattern))

# ── 4-subplot figure ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(24, 12))
fig.suptitle(
    f'Index {index}  |  {row[0]}  |  SG {spacegroup_number} ({row[4]})  '
    f'|  Predicted: {pred_class}   True ext: {true_ext}',
    fontsize=11,
)

draw_xrd_attn(axes[0, 0], two_theta, intensity_pattern, sal_profile,
              twotheta_peaks, hkls, n_labels=10,
              title='Saliency — top-10 HKL labels')

draw_xrd_attn(axes[0, 1], two_theta, intensity_pattern, sal_profile,
              twotheta_peaks, hkls, n_labels=0,
              title='Saliency — all HKL labels  (full)')

draw_xrd_attn(axes[1, 0], two_theta, intensity_pattern, gcam_profile,
              twotheta_peaks, hkls, n_labels=10,
              title='Grad-CAM — top-10 HKL labels')

draw_xrd_attn(axes[1, 1], two_theta, intensity_pattern, gcam_profile,
              twotheta_peaks, hkls, n_labels=0,
              title='Grad-CAM — all HKL labels  (full)')

plt.tight_layout()
plt.show()


c:\Users\egf3\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'torch' is not defined

---
## Utility Code — run once on kernel start, then collapse

Imports, DB setup, metadata, physics helpers, and model loading live here.
You only need to re-run these after a kernel restart.

In [ ]:
import sqlite3
import csv
import pickle
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import spglib
import pyxtal
from math import sin, radians, asin, degrees, ceil, tan
from cctbx import crystal
from cctbx import uctbx
import torch
import torch.nn as nn
from collections import OrderedDict
from resnet_model import ResnetClassifier


In [ ]:
conn   = sqlite3.connect('intrep_crystal_xrd.db')
cursor = conn.cursor()
cursor.execute('DROP TABLE IF EXISTS Structures')

structure_csv = '1k_structures.csv'
with open(structure_csv, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row_csv in reader:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS Structures (
                structure_id, source, generation_commit,
                extinction_group, crystal_system, space_group,
                composition, lattice_a, lattice_b, lattice_c,
                alpha, beta, gamma, atom_sites_json
            )
        """)
        cursor.execute("""
            INSERT INTO Structures (
                structure_id, source, generation_commit, extinction_group,
                crystal_system, space_group, composition,
                lattice_a, lattice_b, lattice_c, alpha, beta, gamma, atom_sites_json
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            row_csv['structure_id'], row_csv['source'],
            row_csv['generation_commit'], int(row_csv['extinction_group']),
            row_csv['crystal_system'],    int(row_csv['space_group']),
            row_csv['composition'],
            float(row_csv['lattice_a']),  float(row_csv['lattice_b']),
            float(row_csv['lattice_c']),  float(row_csv['alpha']),
            float(row_csv['beta']),       float(row_csv['gamma']),
            row_csv['atoms'],
        ))

conn.commit()
conn.close()
print('DB ready.')


DB ready.


In [5]:
hdf5_to_structure = {}
with open('interp_metadata_clean.csv', newline='') as f:
    reader = csv.DictReader(
        row for row in f if not row.lstrip().startswith('#')
    )
    for row in reader:
        hdf5_to_structure[int(row['hdf5_index'])] = row['structure_id']
print(f'Loaded {len(hdf5_to_structure)} index→structure mappings.')


NameError: name 'csv' is not defined

In [6]:
def twotheta_to_d(twotheta, wavelength=1.541838):
    return wavelength / (2 * sin(radians(twotheta) / 2))

def d_to_twotheta(d, wavelength=1.541838):
    return 2 * degrees(asin(wavelength / (2 * d)))

def calcgaussian(x, fwhm):
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return np.exp(-0.5 * ((x / sigma) ** 2))


class Peak:
    scaleFactor = 1
    def __init__(self, center, u, v, w, I,
                 hkl=(None, None, None), shape='Gaussian', eta=0):
        self.center = center
        self.u, self.v, self.w = u, v, w
        self.I = I; self.shape = shape; self.eta = eta; self.hkl = hkl
        try:
            self.H     = np.sqrt(u * (tan(radians(center / 2)) ** 2)
                                 + v * tan(radians(center / 2)) + w)
            self.scale = I * Peak.scaleFactor
        except ValueError:
            self.H = 0; self.scale = 0

    def __call__(self, x):
        if self.shape.lower() == 'gaussian':
            return self.scale * calcgaussian(x - self.center, self.H)
        raise ValueError(f'Unsupported shape: {self.shape}')

    def add(self, v, x):
        idx = (x > self.center - self.H * 3) & (x < self.center + self.H * 3)
        try:
            v[idx] += self.__call__(np.array(x)[idx])
        except Exception:
            pass


def generate_diffraction_pattern_helper(
    spacegroup_number, unit_cell, wavelength,
    twotheta_min, twotheta_max, twothstep, u, v, w, peak_scale=1000,
):
    symmetry   = crystal.symmetry(unit_cell=unit_cell, space_group=spacegroup_number)
    d_max      = twotheta_to_d(twotheta_min, wavelength=wavelength)
    d_min      = twotheta_to_d(twotheta_max, wavelength=wavelength)
    miller_set = symmetry.build_miller_set(anomalous_flag=False, d_min=d_min, d_max=d_max)
    uc         = symmetry.unit_cell()
    twotheta   = np.arange(twotheta_min, twotheta_max, twothstep)
    I          = np.zeros(len(twotheta))
    for hkl in miller_set.indices():
        d_spc     = uc.d(hkl)
        tt_center = 2 * np.arcsin(wavelength / (2 * d_spc)) * 180 / np.pi
        ps        = np.random.uniform(5, peak_scale)
        I        += Peak(tt_center, u, v, w, ps, hkl=hkl)(twotheta)
    lorentz = 1 / (np.sin(np.radians(twotheta)) * np.sin(np.radians(twotheta / 2)))
    I      *= lorentz
    I       = 1000 * I / I.max()
    return twotheta, I


def get_reflections(spacegroup_number, unit_cell,
                    twotheta_min=5.0, twotheta_max=90.0, wavelength=1.541838):
    symm       = crystal.symmetry(unit_cell=unit_cell, space_group=spacegroup_number)
    d_max      = twotheta_to_d(twotheta_min, wavelength)
    d_min      = twotheta_to_d(twotheta_max, wavelength)
    miller_set = symm.build_miller_set(anomalous_flag=False, d_min=d_min, d_max=d_max)
    uc         = symm.unit_cell()
    return np.array(sorted(
        d_to_twotheta(uc.d(hkl), wavelength) for hkl in miller_set.indices()
    ))


def get_reflections_with_hkl(spacegroup_number, unit_cell,
                              twotheta_min=5.0, twotheta_max=90.0, wavelength=1.541838):
    symm       = crystal.symmetry(unit_cell=unit_cell, space_group=spacegroup_number)
    d_max      = twotheta_to_d(twotheta_min, wavelength)
    d_min      = twotheta_to_d(twotheta_max, wavelength)
    miller_set = symm.build_miller_set(anomalous_flag=False, d_min=d_min, d_max=d_max)
    uc         = symm.unit_cell()
    results    = sorted(
        [(d_to_twotheta(uc.d(hkl), wavelength), hkl) for hkl in miller_set.indices()],
        key=lambda x: x[0],
    )
    return np.array([r[0] for r in results]), [r[1] for r in results]


def lattice_letter_from_sgnum(spacegroup_number):
    t = spglib.get_spacegroup_type(spacegroup_number)
    return t['international_short'][0].upper()


def bravais_allows(h, k, l, lattice, r_setting='hex'):
    lattice = lattice.upper()
    if lattice == 'P': return True
    if lattice == 'I': return (h + k + l) % 2 == 0
    if lattice == 'F': return h % 2 == k % 2 == l % 2
    if lattice == 'A': return (k + l) % 2 == 0
    if lattice == 'B': return (h + l) % 2 == 0
    if lattice == 'C': return (h + k) % 2 == 0
    if lattice == 'R':
        if r_setting != 'hex':
            raise ValueError('R lattice assumes hex axes.')
        return (-h + k + l) % 3 == 0
    raise ValueError(f'Unknown lattice type: {lattice}')


def get_reflections_bravais_only(
    spacegroup_number, unit_cell,
    twotheta_min=5.0, twotheta_max=90.0,
    wavelength=1.541838, r_setting='hex', dedupe_tol_deg=1e-4,
):
    uc      = uctbx.unit_cell(unit_cell)
    d_max   = twotheta_to_d(twotheta_min, wavelength)
    d_min   = twotheta_to_d(twotheta_max, wavelength)
    lattice = lattice_letter_from_sgnum(spacegroup_number)
    a_star, b_star, c_star, *_ = uc.reciprocal_parameters()
    hmax = int(ceil(1.0 / (d_min * a_star)))
    kmax = int(ceil(1.0 / (d_min * b_star)))
    lmax = int(ceil(1.0 / (d_min * c_star)))
    twotheta = []
    for h in range(-hmax, hmax + 1):
        for k in range(-kmax, kmax + 1):
            for l in range(-lmax, lmax + 1):
                if (h, k, l) == (0, 0, 0): continue
                if not bravais_allows(h, k, l, lattice, r_setting): continue
                d = uc.d((h, k, l))
                if d < d_min or d > d_max: continue
                tt = d_to_twotheta(d, wavelength)
                if twotheta_min <= tt <= twotheta_max:
                    twotheta.append(tt)
    if not twotheta:
        return np.array([])
    twotheta = np.array(sorted(twotheta))
    if dedupe_tol_deg and dedupe_tol_deg > 0:
        keep = [twotheta[0]]
        for x in twotheta[1:]:
            if abs(x - keep[-1]) > dedupe_tol_deg:
                keep.append(x)
        twotheta = np.array(keep)
    return twotheta


In [7]:
spec_length            = 1000
num_classes            = 99
input_dim              = 1
res_dims               = [32, 64, 64, 64]
res_kernel             = [5, 7, 17, 13]
res_stride             = [4, 4, 5, 3]
num_blocks             = [2, 2, 2, 2]
first_kernel_size      = 13
first_stride           = 1
first_pool_kernel_size = 7
first_pool_stride      = 7
checkpoint = r'C:\Users\egf3\Downloads\xrd_model_kiifq20z.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = ResnetClassifier(
    input_dim=input_dim, res_dims=res_dims, res_kernel=res_kernel,
    res_stride=res_stride, num_blocks=num_blocks,
    first_kernel_size=first_kernel_size, first_stride=first_stride,
    first_pool_kernel_size=first_pool_kernel_size,
    first_pool_stride=first_pool_stride, num_classes=num_classes,
)

ckpt       = torch.load(checkpoint, map_location=device)
state_dict = ckpt.get('model', ckpt)
new_sd     = OrderedDict(
    (k.replace('module.', ''), v) for k, v in state_dict.items()
)
model.load_state_dict(new_sd, strict=False)
model.eval()
print('Model loaded.')

# Uncomment to find the correct GradCAM target layer:
# print([n for n, _ in model.named_modules()])


NameError: name 'torch' is not defined